# Historical Data: UI and API Usage
This notebook explains how to work with **historical measurements** for a grid:
- Upload historical data from JSON
- Explore and visualise measurements
- Filter by time, node and phase
- Delete subsets of historical measurements

We’ll first use the **web UI**, then mirror the same via the **API** from Python.


## 1. Using the Web UI
![Menu](images/menu.png)
### 1.1. Open the Historical UI
1. In your browser, go to the home page:
   ```text
   http://localhost:8000/ (X)
2. Click the “Historical” card, or open directly:
   ```text
   You should see the “Historical Data Management” page with three main cards:
   - Upload Historical Data
   - Delete Historical Measurements
   - Available Grids
![Hist. Menu](images/historical_menu.png)

### 1.2. Upload Historical Data (JSON)
On `/historical/ui`:
1. In the **“Upload Historical Data”** card you’ll see:
   - A dashed **drop zone** (“Drop JSON file here…or click to choose”)
   - A disabled **Upload** button
2. Prepare a JSON file that matches the `HistoricalData` schema:
   - Contains `grid_id`
   - Contains node IDs
   - For each node, a list of timestamped measurements (`datetime`, `power_active`, `power_reactive`, `voltage_magnitude`, `voltage_angle`).
   
Example:
```json
{
  "grid_id": "grid_001",
  "historical": [
    {
      "node_id": "NODE_A",
      "measurements": [
        {
          "datetime": "2025-01-15T10:30:00Z",
          "phase": "R",
          "power_active": 123.4,
          "power_reactive": 45.6,
          "voltage_magnitude": 230.1,
          "voltage_angle": -5.2
        },
        {
          "datetime": "2025-01-15T10:31:00Z",
          "phase": "S",
          "power_active": 120.8,
          "power_reactive": 44.9,
          "voltage_magnitude": 229.8,
          "voltage_angle": -5.0
        }
      ]
    },
    {
      "node_id": "NODE_B",
      "measurements": [
        {
          "datetime": "2025-01-15T10:30:00Z",
          "phase": null,
          "power_active": 98.2,
          "power_reactive": 33.1,
          "voltage_magnitude": 231.0,
          "voltage_angle": -4.8
        }
      ]
    }
  ]
}



3. Either:
   - **Drag and drop** your file onto the dashed box, or
   - Click the box and pick your `.json` from disk.
4. Once a file is selected:
   - The label changes from `No file selected` to the file name.
   - The **Upload** button becomes enabled.
5. Click **Upload**.

![Hist. Menu](images/available_meas.png)

   Behind the scenes the UI:
   - Builds a `FormData` with the file and POSTs to:
   
      **POST /historical/**


   The backend:
   - Parses and validates JSON as HistoricalData schema
   - Creates the historical tables if needed
   - Inserts all measurements

      - On success:

         - You see a message like "Historical data for grid 'my_grid_id' inserted successfully."
         - The page reloads so that my_grid_id appears under Available Grids.
      
      - On failure:

         - The message shows the error text returned by the API
         - (invalid JSON, schema error, DB error, etc.).

### 1.3. Explore historical data for one grid
Still on `/historical/ui`, look at the **Available Grids** card:
- It shows a table of `grid_id` values (coming from the `grids` table).
- Each `grid_id` is a link.
1. Click one grid, e.g. `lv_grid_01`.
This opens:
```text
http://localhost:8000/historical/ui/lv_grid_01

You now see a detail page for that grid with:

- A header: 
    - ← Back to Historical (back to /historical/ui)
    - the standard application layout/navigation

- A “Filter Measurements” card with:
    - Start (datetime-local)
    - End (datetime-local)
    - Phases as checkboxes:
        - R, 
        - S,
        - T

    ![Filter Meas.](images/filter_meas.png)

- A “Measurements Table” card with a full table of measurements (for this grid and filter) with:

    - datetime
    - node_id
    - phase
    - power_active
    - power_reactive
    - voltage_magnitude

    ![Meas. Table](images/meas_table.png)

- A “Voltage Magnitude” chart (with a line chart per node) containing:
    - X-axis: datetime
    - Y-axis: voltage_magnitude

    ![Meas. Table](images/voltage_mag.png)

- A “Active Power” chart (with a line chart per node) containing:
    - X-axis: datetime
    - Y-axis: power_active

Internally, the UI:
- Queries "Measurements" with filters you chose
- Builds two sets of Plotly traces:
- one for voltage
- one for active power
- and renders them directly in the browser with Plotly JS.

### 1.4. Filter measurements in the UI
On `/historical/ui/{grid_id}` you can filter the displayed measurements before rendering the table and charts.

Available filters:

- **Start**  
  Lower datetime bound

- **End**  
  Upper datetime bound

- **Phases**  
  One or more of:
  - `R`
  - `S`
  - `T`

After choosing the filters, click **Apply Filters**.

The page reloads with query parameters. For example:

```text
/historical/ui/lv_grid_01?start=2025-07-31T00:00&end=2025-08-01T00:00&phase=R&phase=S

Behaviour:
- If one or more phase checkboxes are selected, only those phases are shown.
- If no phase filter is applied, the page shows all available rows for the selected grid.
- The measurements table and both charts are rebuilt from the filtered dataset.

The charts are built from the filtered measurement table.

For each node and each available phase in the filtered data, the UI creates:
- one Plotly trace for **voltage magnitude**
- one Plotly trace for **active power**

Each trace is labelled using the format:

```text
{node_id} ({phase})

So the charts show series such as:

- NODE_A (R)
- NODE_A (S)
- NODE_B (T)

This makes it possible to compare both nodes and phases directly in the same figure.

### 1.5. Delete historical measurements
On `/historical/ui` there is also a **“Delete Historical Measurements”** card with:
- `grid_id` (required)
- `node_id` (optional)
- `phase` (R/S/T)
- `start` (datetime)
- `end` (datetime)
- **Delete** button

![Filter Meas.](images/delete_his.png)

The backend delete endpoint is:
```text
DELETE /historical/{grid_id}

- Start / end are parsed as datetime.fromisoformat(...). If provided, they restrict datetime >= start and datetime < end.
- node_id filters a single node.
- phase filters a single phase; if omitted, both aggregated and per-phase rows match.

It returns how many rows were deleted.

## 2. Using the API from Python
Now let’s automate some operations from a notebook using `requests`.

In [ ]:
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime
BASE_URL = "http://localhost:8000"  # adapt if needed
def check_response(resp: requests.Response):
    """Raise for HTTP errors and return parsed JSON or raw text."""
    try:
        resp.raise_for_status()
    except requests.HTTPError as e:
        try:
            print("Error payload:", resp.json())
        except Exception:
            print("Raw response:", resp.text)
        raise e
    try:
        return resp.json()
    except Exception:
        return resp.text

### 2.1. Upload historical data (`POST /historical/`)

In [ ]:
def register_historical_from_file(json_path: str | Path):
    """
    Upload a HistoricalData JSON file to POST /historical/.
    """
    json_path = Path(json_path)
    if not json_path.exists():
        raise FileNotFoundError(json_path)
    url = f"{BASE_URL}/historical/"
    with open(json_path, "rb") as f:
        files = {"file": (json_path.name, f, "application/json")}
        resp = requests.post(url, files=files)
    return check_response(resp)
# Example (uncomment and adapt):
# register_historical_from_file("data/historical_lv_grid_01.json")

The JSON must be valid and conform to the HistoricalData schema; otherwise you’ll get:
- 400 for invalid JSON
- 422 for validation errors
- 500 for DB errors

The response includes a success message with the grid_id.

### 2.2. Retrieve historical records (`GET /historical/data/{grid_id}`)
Endpoint:
```text
GET /historical/data/{grid_id}

Query parameters:
- node_id (optional) – filter by node
- start (optional) – ISO datetime, inclusive lower bound
- end (optional) – ISO datetime, exclusive upper bound
- per_phase (bool, default True)
    - True → returns rows where phase is not NULL
    - False → returns aggregated rows where phase is NULL
- phase (optional "R" | "S" | "T")
    - valid only when per_phase=True

The API returns:
- grid_id
- node_id
- per_phase
- phase
- datetime_range
- records
- count

In [ ]:
def get_historical_data(
    grid_id: str,
    node_id: str | None = None,
    start: datetime | None = None,
    end: datetime | None = None,
    per_phase: bool = True,
    phase: str | None = None,
) -> pd.DataFrame:
    """
    Call GET /historical/data/{grid_id} and return a pandas DataFrame of records.
    """
    params: dict[str, str] = {"per_phase": "true" if per_phase else "false"}

    if node_id:
        params["node_id"] = node_id
    if start:
        params["start"] = start.isoformat()
    if end:
        params["end"] = end.isoformat()
    if phase:
        params["phase"] = phase

    url = f"{BASE_URL}/historical/data/{grid_id}"
    resp = requests.get(url, params=params)
    data = check_response(resp)
    records = data.get("records", [])
    return pd.DataFrame(records)
# Example (after uploading historical data):
# from datetime import datetime
# df = get_historical_data(
#     "lv_grid_01",
#     node_id="N001",
#     start=datetime(2025, 7, 31, 0, 0, 0),
#     end=datetime(2025, 8, 1, 0, 0, 0),
#     per_phase=True,
#     phase="R",
# )
# df.head()

Error cases:
- Invalid datetime format: 400 with a helpful message.
- phase provided with per_phase=false: 400 ("Parameter 'phase' requires per_phase=true.")
- No records matching filters: 404 ("No measurements found matching the criteria.")

### 2.3. Delete historical measurements (`DELETE /historical/{grid_id}`)
Endpoint:
```text
DELETE /historical/{grid_id}

Query parameters (all optional except grid_id):
- start – ISO datetime string (inclusive)
- end – ISO datetime string (exclusive)
- node_id – restrict deletion to one node
- phase – restrict to one phase ("R", "S", "T")

Response example:
```json
{
  "message": "720 measurement(s) deleted for grid 'lv_grid_01'."
}

In [ ]:
def delete_historical_measurements(
    grid_id: str,
    start: datetime | None = None,
    end: datetime | None = None,
    node_id: str | None = None,
    phase: str | None = None,
):
    """
    Call DELETE /historical/{grid_id} with optional filters.
    """
    params: dict[str, str] = {}
    if start:
        params["start"] = start.isoformat()
    if end:
        params["end"] = end.isoformat()
    if node_id:
        params["node_id"] = node_id
    if phase:
        params["phase"] = phase
    url = f"{BASE_URL}/historical/{grid_id}"
    resp = requests.delete(url, params=params)
    return check_response(resp)
# Example: delete all R-phase measurements on node N001 in a time window
# delete_historical_measurements(
#     "lv_grid_01",
#     node_id="N001",
#     phase="R",
#     start=datetime(2025, 7, 31, 0, 0, 0),
#     end=datetime(2025, 8, 1, 0, 0, 0),
# )

### 2.4. Quick plotting in Python (optional)
You can also work with the returned records directly in Python. For example, fetch per-phase measurements for one node and plot voltage magnitude over time with `pandas` and `matplotlib`.

In [ ]:
import matplotlib.pyplot as plt

df = get_historical_data("lv_grid_01", node_id="N001", per_phase=True)
df["datetime"] = pd.to_datetime(df["datetime"])

plt.figure()
for phase in sorted(df["phase"].dropna().unique()):
    subset = df[df["phase"] == phase]
    plt.plot(subset["datetime"], subset["voltage_magnitude"], label=f"Phase {phase}")

plt.xlabel("Datetime")
plt.ylabel("Voltage Magnitude")
plt.legend()
plt.show()

This complements the web UI visualisation when you want to script analysis or export to reports.

## 3. End-to-end workflow
A typical workflow combining UI and API:
1. **Upload historical data (UI)**  
   - Go to `/historical/ui`  
   - Drop your `HistoricalData` JSON and click **Upload**  
   - Confirm the grid appears in **Available Grids**
2. **Explore visually (UI)**  
   - Click a grid ID to open `/historical/ui/{grid_id}`
   - Filter by time window and by one or more phases  
   - Inspect the measurements table and Plotly charts
3. **Scripted analysis (API + Python)**
   - Retrieve rows from `GET /historical/data/{grid_id}`
   - Load them into a `pandas` DataFrame
   - Analyse or plot them in the notebook

In [ ]:
from datetime import datetime
# Fetch measurements
df = get_historical_data(
    "lv_grid_01",
    start=datetime(2025, 7, 31),
    end=datetime(2025, 8, 1),
    per_phase=True,
)
# Do whatever analysis you want on df...
df.head()

4. **Clean up bad imports or subsets**
   - Call `DELETE /historical/{grid_id}` with optional filters such as:
     - `node_id`
     - `phase`
     - `start`
     - `end`

In [ ]:
# Delete a bad import window for one node
delete_historical_measurements(
    "lv_grid_01",
    node_id="N001",
    start=datetime(2025, 7, 31),
    end=datetime(2025, 7, 31, 6),
)

That covers the historical router: upload → explore → query → delete, via both UI and API.

---